# CNEPU Daily - Exploratory Data Analysis

This notebook explores the China News-based Economic Policy Uncertainty (CNEPU) Daily dataset.

**Dataset:** `cnepu-daily.xlsx`

**Description:** Daily Economic Policy Uncertainty index for China from 2000 onwards.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

DATA_PATH = Path('../../datasets/raw/cnepu-daily.xlsx')
CSV_OUTPUT_PATH = Path('../../datasets/raw/cnepu-daily.csv')

## 1. Load and Clean Data

In [ ]:
# Load Excel file
df = pd.read_excel(DATA_PATH)

print("Raw data (first 10 rows):")
display(df.head(10))

# Clean column names
df.columns = df.columns.str.strip()

# Parse date
df['Date'] = pd.to_datetime(df['Date'], format='%Y.%m.%d')

# Sort by date
df = df.sort_values('Date').reset_index(drop=True)

# Extract year, month, day
df['year'] = df['Date'].dt.year
df['month'] = df['Date'].dt.month
df['day'] = df['Date'].dt.day

print("\nCleaned dataset:")
display(df.head())
print("\nData types:")
print(df.dtypes)

## 2. Save to CSV

In [ ]:
# Save cleaned data to CSV
df.to_csv(CSV_OUTPUT_PATH, index=False)
print(f"✓ Data saved to: {CSV_OUTPUT_PATH}")
print(f"  Rows saved: {len(df)}")

## 3. Data Overview

In [ ]:
print("Dataset Shape:", df.shape)
print("\nDate range:", df['Date'].min(), "to", df['Date'].max())
print("Total days:", len(df))
print("\nDataset Info:")
df.info()

## 4. Missing Values

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Percentage': missing_pct})
print("Missing Values Summary:")
display(missing_df[missing_df['Missing Count'] > 0])
if missing.sum() == 0:
    print("\n✓ No missing values found!")

## 5. Descriptive Statistics

In [ ]:
print("Descriptive Statistics for CNEPU_Daily:")
display(df['CNEPU_Daily'].describe())

print("\nAdditional Statistics:")
print(f"Skewness: {df['CNEPU_Daily'].skew():.4f}")
print(f"Kurtosis: {df['CNEPU_Daily'].kurtosis():.4f}")

## 6. Time Series Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(df['Date'], df['CNEPU_Daily'], linewidth=0.5, alpha=0.7)
ax.set_title('China News-based EPU Index - Daily (2000-Present)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('CNEPU Index', fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# With moving averages
df['MA_30d'] = df['CNEPU_Daily'].rolling(window=30, center=True).mean()
df['MA_365d'] = df['CNEPU_Daily'].rolling(window=365, center=True).mean()

fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(df['Date'], df['CNEPU_Daily'], linewidth=0.3, alpha=0.4, label='Daily')
ax.plot(df['Date'], df['MA_30d'], linewidth=1.5, label='30-day MA', color='orange')
ax.plot(df['Date'], df['MA_365d'], linewidth=2, label='365-day MA', color='red')
ax.set_title('CNEPU Index with Moving Averages', fontsize=14, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('CNEPU Index', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['CNEPU_Daily'], bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(df['CNEPU_Daily'].mean(), color='red', linestyle='--', label=f'Mean: {df["CNEPU_Daily"].mean():.2f}')
axes[0].axvline(df['CNEPU_Daily'].median(), color='green', linestyle='--', label=f'Median: {df["CNEPU_Daily"].median():.2f}')
axes[0].set_title('Distribution of CNEPU Index', fontsize=12, fontweight='bold')
axes[0].set_xlabel('CNEPU Index')
axes[0].set_ylabel('Frequency')
axes[0].legend()

axes[1].boxplot(df['CNEPU_Daily'], vert=True)
axes[1].set_title('Box Plot of CNEPU Index', fontsize=12, fontweight='bold')
axes[1].set_ylabel('CNEPU Index')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Yearly Patterns

In [ ]:
yearly_stats = df.groupby('year')['CNEPU_Daily'].agg(['mean', 'std', 'min', 'max', 'count'])
print("Yearly Statistics:")
display(yearly_stats.tail(10))

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(yearly_stats.index, yearly_stats['mean'], alpha=0.7, edgecolor='black')
ax.set_title('Average CNEPU Index by Year', fontsize=14, fontweight='bold')
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Average CNEPU Index', fontsize=12)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 9. Key Events

In [ ]:
top_20 = df.nlargest(20, 'CNEPU_Daily')[['Date', 'CNEPU_Daily']]
print("Top 20 Highest Uncertainty Days:")
display(top_20)

## 10. Summary

In [ ]:
print("=" * 60)
print("KEY FINDINGS - CNEPU DAILY")
print("=" * 60)
print(f"\n1. Dataset Coverage:")
print(f"   - Start: {df['Date'].min().strftime('%Y-%m-%d')}")
print(f"   - End: {df['Date'].max().strftime('%Y-%m-%d')}")
print(f"   - Total Days: {len(df):,}")
print(f"\n2. CNEPU Index Statistics:")
print(f"   - Mean: {df['CNEPU_Daily'].mean():.2f}")
print(f"   - Median: {df['CNEPU_Daily'].median():.2f}")
print(f"   - Std Dev: {df['CNEPU_Daily'].std():.2f}")
print(f"   - Min: {df['CNEPU_Daily'].min():.2f}")
print(f"   - Max: {df['CNEPU_Daily'].max():.2f}")
print(f"\n3. Temporal Insights:")
print(f"   - Year with highest avg: {yearly_stats['mean'].idxmax()} ({yearly_stats['mean'].max():.2f})")
print(f"   - Year with lowest avg: {yearly_stats['mean'].idxmin()} ({yearly_stats['mean'].min():.2f})")
print(f"\n4. Data Quality:")
print(f"   - Missing Values: {df.isnull().sum().sum()}")
print(f"   - CSV file saved: {CSV_OUTPUT_PATH}")
print("\n" + "=" * 60)